In [31]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm


class SelfAttentionBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)
        self.scale = dim ** -0.5

    def forward(self, x):
        # x: [B, D]
        q = self.q_proj(x)  # [B, D]
        k = self.k_proj(x)  # [B, D]
        v = self.v_proj(x)  # [B, D]
        
        attn_score = (q * k).sum(dim=-1, keepdim=True) * self.scale  # [B, 1]
        attn_weight = torch.sigmoid(attn_score)  # [B, 1]
        
        out = v * attn_weight  # [B, D]
        out = self.out_proj(out)
        return out



def fuse_features(quantum, classical, method='concat'):
    """
    quantum: [N, Dq]
    classical: [N, Dc]
    """
    quantum = torch.tensor(quantum, dtype=torch.float32)
    classical = torch.tensor(classical, dtype=torch.float32)
    
    assert quantum.shape[0] == classical.shape[0], "Mismatch in sample count"
    fused = []

    if method == 'add':
        # Require same dimensionality
        min_dim = min(quantum.shape[1], classical.shape[1])
        q = quantum[:, :min_dim]
        c = classical[:, :min_dim]
        fused = q + c

    elif method == 'concat':
        fused = torch.cat([quantum, classical], dim=1)

    elif method == 'attention':
        common_dim = min(quantum.shape[1], classical.shape[1])
        q = quantum[:, :common_dim]
        c = classical[:, :common_dim]
        
        inter = torch.cat([q, c], dim=1)
        
        gate = nn.Sequential(
            nn.Linear(inter.shape[1], inter.shape[1] // 2),
            nn.SiLU(),
            nn.Linear(inter.shape[1] // 2, common_dim),
            nn.Sigmoid()
        )
        attn = gate(inter)
        fused = q * attn + c * (1 - attn)


    elif method == 'self_attention':
        inter = torch.cat([quantum, classical], dim=1)
        sa_block = SelfAttentionBlock(dim=inter.shape[1])
        sa_block.eval()
        with torch.no_grad():
            fused = sa_block(inter)

    elif method == 'dot_product':
        min_dim = min(quantum.shape[1], classical.shape[1])
        q = quantum[:, :min_dim]
        c = classical[:, :min_dim]
        q_norm = F.normalize(q, p=2, dim=1)
        c_norm = F.normalize(c, p=2, dim=1)
        sim = torch.sum(q_norm * c_norm, dim=1, keepdim=True)  # [N, 1]
        weight = (sim + 1) / 2  # scale from [-1,1] to [0,1]
        fused = q * weight + c * (1 - weight)


    return fused
    # return fused.detach().numpy() # for attention fusion_type

if __name__ == '__main__':
    import os
    import pandas as pd

    fusion_type = 'self-attention'

    quantum_path = '/home/eflammere/HybridEmbeddingsForBreastCancerDetection/embeddings/quantum'
    classical_path = '/home/eflammere/HybridEmbeddingsForBreastCancerDetection/embeddings/classical'
    save_path = f'/home/eflammere/HybridEmbeddingsForBreastCancerDetection/embeddings/fused/{fusion_type}'

    if not os.path.exists(save_path):
        os.makedirs(save_path)

    quantum_feat = np.load(os.path.join(quantum_path, 'features.npy'))  # (546, 196)
    classical_feat = np.load(os.path.join(classical_path, 'features.npy'))  # (546, 512)
    labels = np.load(os.path.join(classical_path, 'labels.npy'))  # (546, 1)

    fused = fuse_features(quantum_feat, classical_feat, method=fusion_type)

    np.save(os.path.join(save_path, 'features.npy'), fused)
    np.save(os.path.join(save_path, 'labels.npy'), labels)

    # Save to CSV
    df = pd.DataFrame(fused)
    df['label'] = labels.flatten()
    df.to_csv(os.path.join(save_path, 'features.csv'), index=False)

    print(f">>> Fusion done with method '{fusion_type}'. Saved to {save_path}")


>>> Fusion done with method 'self-attention'. Saved to /home/eflammere/HybridEmbeddingsForBreastCancerDetection/embeddings/fused/self-attention
